# Supplementary File 2: Structured channel pruning, recovery training and inference benchmarking of YOLOv11n.

In [1]:
# Import the libraries used for training, pruning and benchmarking.
from pathlib import Path
from copy import deepcopy
from datetime import datetime
from tempfile import TemporaryDirectory
import gc
import shutil
import torch
import pandas as pd
from IPython.display import display
from ultralytics import YOLO
from ultralytics.nn.modules.block import Bottleneck
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils import ASSETS
from ultralytics.utils.torch_utils import get_flops


In [2]:
# Set the augmented dataset, saved 100-epoch checkpoint and one pruning ratio.
DOWNLOADS = Path.home() / "Downloads"
DATA = DOWNLOADS / "insect.v4-aug.yolov11" / "data.yaml"
BASE_WEIGHTS = DOWNLOADS / "INSECTEGG_benchmark_20261103_130047_701853" / "training" / "augmented" / "weights" / "best.pt"
TRAIN_BASE = False
PRUNING_RATIO = 0.125
IMGSZ = 640
BATCH = 16
WORKERS = 0
DEVICE = 0
EXTRA_EPOCHS = 20
FINE_TUNE_LR = 0.0001
CPU_THREADS = 8
OUT = DOWNLOADS / ("INSECTEGG_Four_Stages_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
OUT.mkdir(parents=True)
A = OUT / "A_base100.pt"
B = OUT / "B_pruned100.pt"
C = OUT / "C_pruned120.pt"
D = OUT / "D_base120.pt"
RESULTS = []
MODEL_INFO = []
print("Dataset:", DATA)
print("Results:", OUT)


Dataset: C:\Users\DR. KAMALDEEP\Downloads\insect.v4-aug.yolov11\data.yaml
Results: C:\Users\DR. KAMALDEEP\Downloads\INSECTEGG_Four_Stages_20261103_230023


In [3]:
# Save each model in the same FP16 checkpoint format.
def save_checkpoint(model, path):
    saved = deepcopy(model).cpu().half().eval()
    if hasattr(saved, "criterion"):
        del saved.criterion
    for parameter in saved.parameters():
        parameter.requires_grad_(False)
    torch.save({"model": saved, "ema": None, "optimizer": None,
                "epoch": -1, "train_args": dict(saved.args), "version": "8.3.208"}, path)

def model_details(checkpoint):
    yolo = YOLO(str(checkpoint))
    model = deepcopy(yolo.model).cpu().float().eval()
    unfused = sum(p.numel() for p in model.parameters())
    model.fuse(verbose=False)
    return {"unfused_parameters": unfused,
            "fused_parameters": sum(p.numel() for p in model.parameters()),
            "GFLOPs_at_640": float(get_flops(model, imgsz=IMGSZ)),
            "checkpoint_MB": checkpoint.stat().st_size / 1e6}


In [4]:
# Physically remove low-L1 hidden channels from Bottleneck layers in groups of eight.
def slice_conv(conv, keep_out=None, keep_in=None):
    keep_out = list(range(conv.out_channels)) if keep_out is None else keep_out
    keep_in = list(range(conv.in_channels)) if keep_in is None else keep_in
    new = torch.nn.Conv2d(len(keep_in), len(keep_out), conv.kernel_size,
                          conv.stride, conv.padding, conv.dilation,
                          bias=conv.bias is not None).to(conv.weight)
    with torch.no_grad():
        new.weight.copy_(conv.weight[keep_out][:, keep_in])
        if conv.bias is not None:
            new.bias.copy_(conv.bias[keep_out])
    return new

def prune_model(model, ratio):
    model = deepcopy(model).cpu().float()
    removed_total = 0
    eligible_total = 0
    for layer in list(model.modules()):
        if type(layer) is not Bottleneck:
            continue
        first, second, bn = layer.cv1.conv, layer.cv2.conv, layer.cv1.bn
        width = first.out_channels
        eligible_total += width
        remove = max(0, min(int(width * ratio) // 8 * 8, width - 8))
        if remove == 0:
            continue
        scores = first.weight.detach().abs().sum(dim=(1, 2, 3))
        keep = scores.argsort(stable=True)[remove:].sort().values.tolist()
        layer.cv1.conv = slice_conv(first, keep_out=keep)
        layer.cv2.conv = slice_conv(second, keep_in=keep)
        new_bn = torch.nn.BatchNorm2d(len(keep), eps=bn.eps, momentum=bn.momentum).to(bn.weight)
        with torch.no_grad():
            new_bn.weight.copy_(bn.weight[keep])
            new_bn.bias.copy_(bn.bias[keep])
            new_bn.running_mean.copy_(bn.running_mean[keep])
            new_bn.running_var.copy_(bn.running_var[keep])
            new_bn.num_batches_tracked.copy_(bn.num_batches_tracked)
        layer.cv1.bn = new_bn
        removed_total += remove
    return model, removed_total, eligible_total


In [5]:
# Fine-tune either branch for 20 epochs using AdamW, lr=0.0001 and no mosaic.
class KeepArchitectureTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = deepcopy(weights).float()
        for parameter in model.parameters():
            parameter.requires_grad_(True)
        return model

def fine_tune(source, target):
    model = YOLO(str(source))
    parameter_count = sum(p.numel() for p in model.model.parameters())
    with TemporaryDirectory(prefix="eggplant_train_", ignore_cleanup_errors=True) as work:
        model.train(trainer=KeepArchitectureTrainer, data=str(DATA),
                    epochs=EXTRA_EPOCHS, patience=EXTRA_EPOCHS,
                    batch=BATCH, imgsz=IMGSZ, device=DEVICE, workers=WORKERS,
                    optimizer="AdamW", lr0=FINE_TUNE_LR, lrf=0.01,
                    momentum=0.9, weight_decay=0.0005,
                    warmup_epochs=0, warmup_bias_lr=FINE_TUNE_LR,
                    mosaic=0, close_mosaic=0, seed=0, deterministic=True,
                    amp=True, resume=False, plots=False,
                    project=work, name="train")
        history = pd.read_csv(Path(model.trainer.save_dir) / "results.csv")
        last = YOLO(str(model.trainer.last))
        assert len(history) == EXTRA_EPOCHS, "The additional training did not finish."
        assert sum(p.numel() for p in last.model.parameters()) == parameter_count, "Training changed the architecture."
        save_checkpoint(last.model, target)
        history.to_csv(OUT / (target.stem + "_training.csv"), index=False)
        del last, model
        gc.collect()
        torch.cuda.empty_cache()


In [6]:
# Benchmark all stages with validation-mean timing: batch 1, 640 pixels and train-calibrated INT8.
FORMATS = [
    ("PyTorch FP32 GPU", "pt", 0, False, False),
    ("PyTorch FP16 GPU", "pt", 0, True, False),
    ("ONNX FP32 GPU", "onnx", 0, False, False),
    ("ONNX FP16 GPU", "onnx", 0, True, False),
    ("ONNX FP32 CPU", "onnx", "cpu", False, False),
    ("TensorRT FP32 GPU", "engine", 0, False, False),
    ("TensorRT FP16 GPU", "engine", 0, True, False),
    ("TensorRT INT8 GPU", "engine", 0, False, True),
]

def benchmark_stage(checkpoint, stage, removed_channels=0):
    global RESULTS, MODEL_INFO
    torch.set_num_threads(CPU_THREADS)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    details = model_details(checkpoint)
    MODEL_INFO = [row for row in MODEL_INFO if row["stage"] != stage]
    MODEL_INFO.append({"stage": stage, "removed_channels": removed_channels, **details})
    RESULTS = [row for row in RESULTS if row["stage"] != stage]
    with TemporaryDirectory(prefix="eggplant_benchmark_", ignore_cleanup_errors=True) as work:
        local = Path(work) / "model.pt"
        shutil.copy2(checkpoint, local)
        for label, fmt, device, half, int8 in FORMATS:
            model = YOLO(str(local))
            artifact = str(local) if fmt == "pt" else model.export(
                format=fmt, imgsz=IMGSZ, batch=1, device=device,
                half=half, int8=int8, dynamic=False, simplify=True,
                nms=False, data=str(DATA), split="train", verbose=False)
            exported = YOLO(str(artifact), task="detect")
            exported.predict(ASSETS / "bus.jpg", imgsz=IMGSZ, device=device,
                             half=half, save=False, verbose=False)
            metrics = exported.val(data=str(DATA), split="test", batch=1,
                imgsz=IMGSZ, device=device, half=half, int8=int8,
                workers=WORKERS, conf=0.001, iou=0.7,
                plots=False, save_json=False, verbose=False,
                project=work, name="validation", exist_ok=True)
            p, r = float(metrics.box.mp), float(metrics.box.mr)
            inference_ms = float(metrics.speed["inference"])
            RESULTS.append({"stage": stage, "format": label,
                "mAP50_percent": float(metrics.box.map50) * 100,
                "mAP50_95_percent": float(metrics.box.map) * 100,
                "precision_percent": p * 100, "recall_percent": r * 100,
                "F1_percent": 200 * p * r / (p + r) if p + r else 0,
                "inference_ms": inference_ms, "FPS": 1000 / (inference_ms + 0.001),
                "export_MB": Path(artifact).stat().st_size / 1e6,
                "removed_channels": removed_channels, **details})
            pd.DataFrame(RESULTS).to_csv(OUT / "results.csv", index=False)
            del exported, model, metrics
            gc.collect()
            torch.cuda.empty_cache()
    display(pd.DataFrame([row for row in RESULTS if row["stage"] == stage]).round(4))


In [7]:
# A: load best.pt from the completed 100-epoch run, or train that run if TRAIN_BASE=True.
if TRAIN_BASE:
    with TemporaryDirectory(prefix="eggplant_base_", ignore_cleanup_errors=True) as work:
        model = YOLO("yolo11n.pt")
        model.train(data=str(DATA), epochs=100, batch=BATCH, imgsz=IMGSZ,
                    device=DEVICE, workers=WORKERS, seed=0,
                    plots=False, project=work, name="train")
        best = YOLO(str(model.trainer.best))
        save_checkpoint(best.model, A)
        pd.read_csv(Path(model.trainer.save_dir) / "results.csv").to_csv(OUT / "A_base100_training.csv", index=False)
        del model, best
        gc.collect()
        torch.cuda.empty_cache()
else:
    model = YOLO(str(BASE_WEIGHTS))
    save_checkpoint(model.model, A)
    del model


In [8]:
# Stage A: benchmark the base model from the 100-epoch training run.
benchmark_stage(A, "A_base100")


Ultralytics 8.3.208  Python-3.10.14 torch-2.4.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 178.746.9 MB/s, size: 67.2 KB)
val: Scanning C:\Users\DR. KAMALDEEP\Downloads\insect.v4-aug.yolov11\test\labels.cache... 43 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 43/43  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 43/43 45.8it/s 0.9s0.1s
                   all         43         43      0.864      0.886      0.928      0.907
Speed: 0.4ms preprocess, 12.1ms inference, 0.0ms loss, 2.6ms postprocess per image
Ultralytics 8.3.208  Python-3.10.14 torch-2.4.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 837.9238.6 MB/s, size: 65.9 KB)
val: Scanning C:\Users\DR. KAMALDEEP\Downloads\insect.v4-aug.yolov11\test\labels.cache... 43 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 43/43  0.0s
                 Clas

,stage,format,mAP50_percent,mAP50_95_percent,precision_percent,recall_percent,F1_percent,inference_ms,FPS,export_MB,removed_channels,unfused_parameters,fused_parameters,GFLOPs_at_640,checkpoint_MB
0,A_base100,PyTorch FP32 GPU,92.7951,90.6669,86.3712,88.6136,87.4780,12.0606,82.9079,5.4919,0,2591010,2583322,6.3184,5.4919
1,A_base100,PyTorch FP16 GPU,92.7951,90.6669,85.9557,89.7914,87.8317,16.2362,61.5869,5.4919,0,2591010,2583322,6.3184,5.4919
2,A_base100,ONNX FP32 GPU,88.1389,85.7849,84.3304,77.8066,80.9372,7.8963,126.6254,10.6086,0,2591010,2583322,6.3184,5.4919
3,A_base100,ONNX FP16 GPU,88.1389,85.7849,84.3369,77.8246,80.9500,9.6530,103.5845,5.3578,0,2591010,2583322,6.3184,5.4919
4,A_base100,ONNX FP32 CPU,88.1389,85.7849,84.2883,77.8206,80.9255,30.6544,32.6206,10.6086,0,2591010,2583322,6.3184,5.4919
5,A_base100,TensorRT FP32 GPU,88.1389,85.9784,84.3091,77.8320,80.9412,5.7127,175.0169,12.6148,0,2591010,2583322,6.3184,5.4919
6,A_base100,TensorRT FP16 GPU,88.1389,85.9784,84.2685,77.8292,80.9210,5.4804,182.4347,8.4112,0,2591010,2583322,6.3184,5.4919
7,A_base100,TensorRT INT8 GPU,89.7182,87.9527,85.3129,77.9888,81.4866,6.5599,152.4184,4.9271,0,2591010,2583322,6.3184,5.4919


In [9]:
# B: prune a fresh copy of A without extra training.
model = YOLO(str(A))
pruned, REMOVED_CHANNELS, ELIGIBLE_CHANNELS = prune_model(model.model, PRUNING_RATIO)
save_checkpoint(pruned, B)
del model, pruned
print("Removed hidden channels:", REMOVED_CHANNELS, "of", ELIGIBLE_CHANNELS)


Removed hidden channels: 32 of 424


In [10]:
# Stage B: benchmark the pruned model before recovery.
benchmark_stage(B, "B_pruned100", REMOVED_CHANNELS)


Ultralytics 8.3.208  Python-3.10.14 torch-2.4.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 774.6204.7 MB/s, size: 62.6 KB)
val: Scanning C:\Users\DR. KAMALDEEP\Downloads\insect.v4-aug.yolov11\test\labels.cache... 43 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 43/43 42.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 43/43 50.3it/s 0.9s0.1s
                   all         43         43      0.871      0.894      0.908      0.798
Speed: 0.8ms preprocess, 11.6ms inference, 0.0ms loss, 1.3ms postprocess per image
Ultralytics 8.3.208  Python-3.10.14 torch-2.4.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 810.1251.0 MB/s, size: 66.6 KB)
val: Scanning C:\Users\DR. KAMALDEEP\Downloads\insect.v4-aug.yolov11\test\labels.cache... 43 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 43/43 43.0Kit/s 0.0s
  

,stage,format,mAP50_percent,mAP50_95_percent,precision_percent,recall_percent,F1_percent,inference_ms,FPS,export_MB,removed_channels,unfused_parameters,fused_parameters,GFLOPs_at_640,checkpoint_MB
0,B_pruned100,PyTorch FP32 GPU,90.7527,79.8075,87.1289,89.4482,88.2733,11.5624,86.4795,5.4200,32,2554082,2546426,6.2889,5.42
1,B_pruned100,PyTorch FP16 GPU,90.7527,79.0025,87.0888,89.4126,88.2354,11.9148,83.9223,5.4200,32,2554082,2546426,6.2889,5.42
2,B_pruned100,ONNX FP32 GPU,89.0311,79.1201,81.3730,82.9484,82.1531,10.4432,95.7465,10.4610,32,2554082,2546426,6.2889,5.42
3,B_pruned100,ONNX FP16 GPU,89.0382,79.1293,81.3803,82.9314,82.1485,11.1808,89.4310,5.2841,32,2554082,2546426,6.2889,5.42
4,B_pruned100,ONNX FP32 CPU,89.0311,79.5285,81.3881,82.9506,82.1619,32.3879,30.8748,10.4610,32,2554082,2546426,6.2889,5.42
5,B_pruned100,TensorRT FP32 GPU,89.0311,79.1201,81.3853,82.9377,82.1542,5.4433,183.6781,12.9114,32,2554082,2546426,6.2889,5.42
6,B_pruned100,TensorRT FP16 GPU,89.0311,79.1201,81.4107,82.9055,82.1513,7.1503,139.8348,8.6094,32,2554082,2546426,6.2889,5.42
7,B_pruned100,TensorRT INT8 GPU,88.7918,76.2537,82.4625,78.5413,80.4542,5.8758,170.1598,4.8602,32,2554082,2546426,6.2889,5.42


In [11]:
# C: train B for 20 recovery epochs and save the final checkpoint.
fine_tune(B, C)


New https://pypi.org/project/ultralytics/8.4.201 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.208  Python-3.10.14 torch-2.4.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\DR. KAMALDEEP\Downloads\insect.v4-aug.yolov11\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\DR. KAMALDEEP\Downloads\INSECTEGG_Four_S

In [12]:
# Stage C: benchmark the pruned model after 20 recovery epochs.
benchmark_stage(C, "C_pruned120", REMOVED_CHANNELS)


Ultralytics 8.3.208  Python-3.10.14 torch-2.4.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
val: Fast image access  (ping: 0.10.1 ms, read: 112.235.4 MB/s, size: 64.5 KB)
val: Scanning C:\Users\DR. KAMALDEEP\Downloads\insect.v4-aug.yolov11\test\labels.cache... 43 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 43/43 43.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 43/43 53.0it/s 0.8s0.0s
                   all         43         43      0.785      0.865      0.895       0.87
Speed: 0.5ms preprocess, 10.6ms inference, 0.0ms loss, 1.2ms postprocess per image
Ultralytics 8.3.208  Python-3.10.14 torch-2.4.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 652.4210.2 MB/s, size: 66.6 KB)
val: Scanning C:\Users\DR. KAMALDEEP\Downloads\insect.v4-aug.yolov11\test\labels.cache... 43 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 43/43 42.7Kit/s 0.0s
   

,stage,format,mAP50_percent,mAP50_95_percent,precision_percent,recall_percent,F1_percent,inference_ms,FPS,export_MB,removed_channels,unfused_parameters,fused_parameters,GFLOPs_at_640,checkpoint_MB
0,C_pruned120,PyTorch FP32 GPU,89.5118,86.9856,78.4870,86.5034,82.3005,10.5780,94.5266,5.4199,32,2554082,2546426,6.2889,5.4199
1,C_pruned120,PyTorch FP16 GPU,89.3377,86.9546,80.0599,84.3888,82.1674,12.8929,77.5562,5.4199,32,2554082,2546426,6.2889,5.4199
2,C_pruned120,ONNX FP32 GPU,86.2800,84.5122,88.6998,72.9618,80.0647,11.5443,86.6156,10.4609,32,2554082,2546426,6.2889,5.4199
3,C_pruned120,ONNX FP16 GPU,86.2800,84.5122,88.6624,72.9673,80.0528,11.3348,88.2157,5.2840,32,2554082,2546426,6.2889,5.4199
4,C_pruned120,ONNX FP32 CPU,86.2800,84.5122,88.7021,72.9391,80.0520,34.3416,29.1183,10.4609,32,2554082,2546426,6.2889,5.4199
5,C_pruned120,TensorRT FP32 GPU,86.2800,85.0620,88.5965,72.9589,80.0209,5.5967,178.6458,12.9215,32,2554082,2546426,6.2889,5.4199
6,C_pruned120,TensorRT FP16 GPU,86.2800,85.0620,88.4810,73.1106,80.0648,5.6134,178.1122,8.8158,32,2554082,2546426,6.2889,5.4199
7,C_pruned120,TensorRT INT8 GPU,84.8263,82.9043,71.9327,85.1359,77.9793,6.7877,147.3046,4.9120,32,2554082,2546426,6.2889,5.4199


In [13]:
# D: independently train A for the same 20 epochs and save the final checkpoint.
fine_tune(A, D)


New https://pypi.org/project/ultralytics/8.4.201 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.208  Python-3.10.14 torch-2.4.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\DR. KAMALDEEP\Downloads\insect.v4-aug.yolov11\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\DR. KAMALDEEP\Downloads\INSECTEGG_Four_S

In [14]:
# Stage D: benchmark the unpruned model after 20 additional epochs.
benchmark_stage(D, "D_base120")


Ultralytics 8.3.208  Python-3.10.14 torch-2.4.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 416.361.2 MB/s, size: 64.5 KB)
val: Scanning C:\Users\DR. KAMALDEEP\Downloads\insect.v4-aug.yolov11\test\labels.cache... 43 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 43/43  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 43/43 51.1it/s 0.8s0.1s
                   all         43         43      0.823      0.821      0.897       0.88
Speed: 0.5ms preprocess, 11.1ms inference, 0.0ms loss, 1.4ms postprocess per image
Ultralytics 8.3.208  Python-3.10.14 torch-2.4.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 540.596.4 MB/s, size: 66.6 KB)
val: Scanning C:\Users\DR. KAMALDEEP\Downloads\insect.v4-aug.yolov11\test\labels.cache... 43 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 43/43 43.0Kit/s 0.0s
             

,stage,format,mAP50_percent,mAP50_95_percent,precision_percent,recall_percent,F1_percent,inference_ms,FPS,export_MB,removed_channels,unfused_parameters,fused_parameters,GFLOPs_at_640,checkpoint_MB
0,D_base120,PyTorch FP32 GPU,89.6945,87.9730,82.3172,82.0590,82.1879,11.0589,90.4168,5.4918,0,2591010,2583322,6.3184,5.4918
1,D_base120,PyTorch FP16 GPU,89.7824,88.0601,82.2971,82.0813,82.1891,11.9147,83.9230,5.4918,0,2591010,2583322,6.3184,5.4918
2,D_base120,ONNX FP32 GPU,87.5754,86.7225,79.9080,77.1460,78.5028,11.3246,88.2957,10.6085,0,2591010,2583322,6.3184,5.4918
3,D_base120,ONNX FP16 GPU,87.5754,86.7225,79.2766,78.5429,78.9081,11.2984,88.5002,5.3578,0,2591010,2583322,6.3184,5.4918
4,D_base120,ONNX FP32 CPU,87.5754,86.7229,79.9028,77.1210,78.4873,33.1675,30.1491,10.6085,0,2591010,2583322,6.3184,5.4918
5,D_base120,TensorRT FP32 GPU,87.5754,86.7229,79.3882,78.4740,78.9285,6.2219,160.6958,13.4000,0,2591010,2583322,6.3184,5.4918
6,D_base120,TensorRT FP16 GPU,87.5754,86.7229,79.8501,77.3509,78.5806,5.4248,184.3033,8.5857,0,2591010,2583322,6.3184,5.4918
7,D_base120,TensorRT INT8 GPU,85.9796,84.9493,70.6688,86.2190,77.6732,6.7414,148.3159,4.8808,0,2591010,2583322,6.3184,5.4918


In [15]:
# Compare pruning, recovery and extra training without mixing timing methods.
results = pd.DataFrame(RESULTS)
comparisons = []
PAIRS = [
    ("A_base100", "B_pruned100", "Pruning only: A to B"),
    ("A_base100", "C_pruned120", "Pruning + recovery: A to C"),
    ("B_pruned100", "C_pruned120", "Recovery effect: B to C"),
    ("A_base100", "D_base120", "Extra training effect: A to D"),
    ("D_base120", "C_pruned120", "Pruning with matched training: D to C"),
]
for reference, candidate, label in PAIRS:
    first = results[results.stage == reference].set_index("format")
    second = results[results.stage == candidate].set_index("format")
    for fmt in first.index:
        a, b = first.loc[fmt], second.loc[fmt]
        comparisons.append({"comparison": label, "format": fmt,
            "reference_FPS": a.FPS, "candidate_FPS": b.FPS,
            "FPS_change_percent": (b.FPS / a.FPS - 1) * 100,
            "mAP50_change_percentage_points": b.mAP50_percent - a.mAP50_percent})
comparisons = pd.DataFrame(comparisons)
display(results.pivot(index="format", columns="stage", values="FPS").round(2))
display(comparisons.round(3))


stage,A_base100,B_pruned100,C_pruned120,D_base120
format,,,,
ONNX FP16 GPU,103.58,89.43,88.22,88.50
ONNX FP32 CPU,32.62,30.87,29.12,30.15
ONNX FP32 GPU,126.63,95.75,86.62,88.30
PyTorch FP16 GPU,61.59,83.92,77.56,83.92
PyTorch FP32 GPU,82.91,86.48,94.53,90.42
TensorRT FP16 GPU,182.43,139.83,178.11,184.30
TensorRT FP32 GPU,175.02,183.68,178.65,160.70
TensorRT INT8 GPU,152.42,170.16,147.30,148.32


,comparison,format,reference_FPS,candidate_FPS,FPS_change_percent,mAP50_change_percentage_points
0,Pruning only: A to B,PyTorch FP32 GPU,82.908,86.480,4.308,-2.042
1,Pruning only: A to B,PyTorch FP16 GPU,61.587,83.922,36.266,-2.042
2,Pruning only: A to B,ONNX FP32 GPU,126.625,95.746,-24.386,0.892
3,Pruning only: A to B,ONNX FP16 GPU,103.585,89.431,-13.664,0.899
4,Pruning only: A to B,ONNX FP32 CPU,32.621,30.875,-5.352,0.892
5,Pruning only: A to B,TensorRT FP32 GPU,175.017,183.678,4.949,0.892
6,Pruning only: A to B,TensorRT FP16 GPU,182.435,139.835,-23.351,0.892
7,Pruning only: A to B,TensorRT INT8 GPU,152.418,170.160,11.640,-0.926
8,Pruning + recovery: A to C,PyTorch FP32 GPU,82.908,94.527,14.014,-3.283
9,Pruning + recovery: A to C,PyTorch FP16 GPU,61.587,77.556,25.930,-3.457


In [16]:
# Save one readable report; checkpoints, training CSVs and results.csv are retained.
def markdown_table(frame):
    def text(value):
        return f"{value:.4f}" if isinstance(value, float) else str(value)
    return "\n".join(["| " + " | ".join(frame.columns) + " |",
                       "| " + " | ".join(["---"] * len(frame.columns)) + " |"] +
                      ["| " + " | ".join(text(v) for v in row) + " |"
                       for row in frame.itertuples(index=False, name=None)])

info = pd.DataFrame(MODEL_INFO)
base_parameters = info.loc[info.stage == "A_base100", "fused_parameters"].iloc[0]
info["parameter_reduction_percent"] = (1 - info.fused_parameters / base_parameters) * 100
fps_table = results.pivot(index="format", columns="stage", values="FPS").reset_index()
map_table = results.pivot(index="format", columns="stage", values="mAP50_percent").reset_index()
report = ["# Four-stage comparison",
    "A: best checkpoint from a completed 100-epoch training run. B: A after physical pruning, before recovery. C: B after 20 recovery epochs. D: A after the same 20 extra epochs. C and D use their final (last.pt) checkpoints, not the best intermediate recovery checkpoint.",
    f"Dataset: {DATA}. Requested local pruning ratio: {PRUNING_RATIO}. Removed: {REMOVED_CHANNELS}/{ELIGIBLE_CHANNELS} eligible hidden channels. The local pruning ratio is not the percentage of all parameters removed.",
    f"All stages: validation-mean inference timing, batch 1, imgsz {IMGSZ}, workers {WORKERS}, CPU threads {CPU_THREADS}, PyTorch TF32 disabled. ONNX/TensorRT exports are static. Backend-default rectangular padding is retained. INT8 calibrates on training images. FPS=1000/(unrounded inference_ms+0.001); preprocessing and NMS are excluded.",
    "Eight supported configurations are measured; TensorRT precision labels describe build modes and may contain mixed precision. No custom timing or 90% accuracy cutoff is used.",
    "## Models", markdown_table(info),
    "## FPS", markdown_table(fps_table),
    "## Test mAP50 (%)", markdown_table(map_table),
    "## Comparisons", markdown_table(comparisons),
    "Positive FPS change means faster measured inference; negative means slower. mAP differences are percentage points. These are recorded measurements, not significance estimates across repeated independent runs.",
    "Full precision, recall, F1, mAP50–95, latency and model-size values are in results.csv. F1 is the harmonic mean of the reported mean precision and recall. Temporary exports and training artifacts were not retained."]
(OUT / "RESULT.md").write_text("\n\n".join(report), encoding="utf-8")
print("Saved:", OUT)
print("Main files: results.csv and RESULT.md")


Saved: C:\Users\DR. KAMALDEEP\Downloads\INSECTEGG_Four_Stages_20261103_230023
Main files: results.csv and RESULT.md
